# Training of the student

What we do:
1) Load the training dataset
2) Run training using the huggingface client
3) Save the model


Student: Qwen2.5-1.5B-Instruct

In [1]:
import dotenv

dotenv.load_dotenv()

True

In [2]:
from core.types import *
from core.utils.huggingface_training_client import HuggingFaceTrainingClient
from core.utils.ollama_inference_client import OllamaInferenceClient
from core.utils.openai_client import OpenAIClient, ProcessingMode
from doom.preprocessing.doom_game_state_perturbator import DoomGameStatePerturbator
from doom.utils.doom_game_state import DoomGameState, MonsterType, WeaponName, AimedAtType
from core.distillation.sampling import stratified_sampling_with_features
from sklearn.cluster import DBSCAN
from transformers import AutoTokenizer
from dataclasses import dataclass, asdict
from collections import Counter
from typing import Iterable
from pathlib import Path
from openai.types.responses import Response as OpenAIResponse

import os
import json
import csv
import numpy as np
import pandas as pd

/home/filippo/gamepals-llm-distillation/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
dataset1_path = Path("data/outputs/selected-data-low-reasoning-gpt5.csv")
dataset2_path = Path("data/outputs/non-selected-data-low-reasoning-gpt5.csv")

df1 = pd.read_csv(dataset1_path)
df2 = pd.read_csv(dataset2_path)
df = pd.concat([df1, df2])
df.dropna(axis=0, how='all', inplace=True)

print(len(df))

# I should immediately filter out my labeling set - it should not be involved in the training
labelled = df[df["selected_for_labelling"] == True]
df = df[df["selected_for_labelling"] != True]

print(len(labelled))
print(len(df))

2872
50
2822


In [5]:
df

,input_id,game_state,command,command_intent,command_explicitness,command_atomicity,command_contextuality,game_actions,latency,reason_if_failed,...,action_full_correct,action_unnecessary,action_imprecise_sequentiality,action_imprecise_parameters,action_harming_sequentiality,action_harming_parameters,action_missing,action_harming,action_wrong_syntax,label
0,state-233-p0-uc0,AIMED_AT:\n type: Wall\n distance: 330.86\n ...,Go hit that switch on the wall,Approach the interactable wall and activate th...,0.85,0.40,0.85,SPRINT 0.0 330.86\nINTERACT,0.0,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
1,state-233-p0-uc1,AIMED_AT:\n type: Wall\n distance: 330.86\n ...,Look around for any new openings,Survey the surroundings for newly revealed pat...,0.60,0.50,0.90,ROTATE 360 0,0.0,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
2,state-233-p0-uc2,AIMED_AT:\n type: Wall\n distance: 330.86\n ...,Keep the pistol ready and move forward,Advance cautiously while staying prepared for ...,0.70,0.50,0.70,SPRINT 0.0 330.86,0.0,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
3,state-233-p1-uc0,AIMED_AT:\n type: Wall\n distance: 330.86\n ...,Go press that switch ahead,Move to the interactable wall in front and act...,0.85,0.40,0.85,SPRINT 0.0 330.86\nINTERACT,0.0,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
4,state-233-p1-uc1,AIMED_AT:\n type: Wall\n distance: 330.86\n ...,Hit the use key on that wall,Trigger interaction with the nearby interactab...,0.90,0.85,0.70,INTERACT,0.0,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2817,state-58188-p0-uc1,AIMED_AT:\n type: Monster\n distance: 301.90...,Back up while firing at the skull,Kite the charging lost soul to stay safe while...,0.80,0.40,0.90,ROTATE_TO_TARGET MONSTER_0\nASYNC FIRE 2.0\nMO...,0.0,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
2818,state-58188-p0-uc2,AIMED_AT:\n type: Monster\n distance: 301.90...,Conserve ammo and just dodge it,Avoid damage from the lost soul without spendi...,0.75,0.50,0.85,SPRINT 100.0 0.0,0.0,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
2819,state-58188-p1-uc0,AIMED_AT:\n type: Monster\n distance: 301.90...,Put that lost soul in my crosshair,Align the aim precisely with the approaching l...,0.85,0.35,0.85,ROTATE_TO_TARGET MONSTER_0,0.0,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
2820,state-58188-p1-uc1,AIMED_AT:\n type: Monster\n distance: 301.90...,Shoot the flying skull now,Immediately fire the pistol to damage the visi...,0.90,0.80,0.80,ROTATE_TO_TARGET MONSTER_0\nFIRE 1.0,0.0,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN


In [6]:
# Extract only <inputs, labels>
# For inputs, we do not want (nor need) the full prompt. Just a couple of general pieces of information.
# That is, the inputs are SMALL PROMPT + GAME_STATE + USER COMMAND
# The labels are the df.labels

dataset = [
    TrainingEntry(
        id=row.input_id,
        game_state=row.game_state,
        user_command=row.command,
        expected_actions=row.game_actions,
        metrics=TrainingEntryMetrics(
            explicitness=row.command_explicitness,
            atomicity=row.command_atomicity,
            contextuality=row.command_contextuality,
            cluster_id=row.cluster_id
        )
    )
    for row in df.itertuples()
]

TypeError: TrainingEntry.__init__() got an unexpected keyword argument 'id'

In [16]:
training_client = HuggingFaceTrainingClient[TrainingEntry](
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device="cuda",
    working_dir=Path("models/huggingface"),
    use_flash_attention_2=True
)

🏋️  Initialized HuggingFaceTrainingClient for Qwen/Qwen2.5-1.5B-Instruct
   Device: cuda
   Flash Attention 2: True


In [17]:
def format_example(example: TrainingEntry, tokenizer: AutoTokenizer) -> str:
    messages = [
        {
            "role": "system",
            "content": "You are a game command parser that converts natural language commands into DSL instructions."
        },
        {
            "role": "user",
            "content": f"Game State: {example.game_state}\nCommand: {example.user_command}"
        },
        {
            "role": "assistant",
            "content": example.expected_actions
        }
    ]

    if hasattr(tokenizer, "apply_chat_template"):
        formatted = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        # Template already adds EOS - just return it
        return formatted.rstrip()

    return json.dumps(messages) + tokenizer.eos_token

In [20]:
# Sample based on the metrics we have for each example. Compute a 80-15-5 split.

remaining_pool, remaining_test = stratified_sampling_with_features(
    dataset,
    eval_ratio=(140 - 50) / len(dataset) # A little tricky because I made the mistake of labeling before splitting. Can be cleaner if done from zero.
)

train_data, val_data = stratified_sampling_with_features(
    remaining_pool,
    eval_ratio=0.15 / 0.95 # This allows to separate the remaining 95% into the 80-15 split
)

# I should save the remaining_test together with the selected for labeling, as I will run tests ONLY on them later
test_ids = [item.input_id for item in remaining_test]

# Save to CSV (one ID per line)
with open('test_split_ids.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    for tid in test_ids:
        writer.writerow([tid])

📊 Stratified sampling by features:
   Total clusters: 10
   Cluster 7.0: 159 train, 9 eval
   Cluster 8.0: 475 train, 24 eval
   Cluster 5.0: 325 train, 16 eval
   Cluster 4.0: 397 train, 16 eval
   Cluster 2.0: 192 train, 9 eval
   Cluster 9.0: 193 train, 11 eval
   Cluster 3.0: 130 train, 6 eval
   Cluster 6.0: 259 train, 14 eval
   Cluster 1.0: 492 train, 23 eval
   Cluster 0.0: 68 train, 4 eval
📊 Stratified sampling by features:
   Total clusters: 10
   Cluster 5.0: 271 train, 54 eval
   Cluster 9.0: 160 train, 33 eval
   Cluster 2.0: 161 train, 31 eval
   Cluster 6.0: 216 train, 43 eval
   Cluster 4.0: 330 train, 67 eval
   Cluster 1.0: 411 train, 81 eval
   Cluster 8.0: 394 train, 81 eval
   Cluster 7.0: 132 train, 27 eval
   Cluster 0.0: 57 train, 11 eval
   Cluster 3.0: 109 train, 21 eval


AttributeError: 'TrainingEntry' object has no attribute 'input_id'

In [11]:
training_client.load_tokenizer()


📦 Loading tokenizer: Qwen/Qwen2.5-1.5B-Instruct
   ✓ Tokenizer loaded successfully!



In [13]:
training_client.fine_tune(
    train_dataset=train_data,
    eval_dataset=val_data,
    format_example=format_example,
    output_dir=Path("models/huggingface/training"),
    num_epochs=5,
    batch_size=4,
    learning_rate=2e-4,
    gradient_accumulation_steps=2,
)

`torch_dtype` is deprecated! Use `dtype` instead!



📦 Loading model for training: Qwen/Qwen2.5-1.5B-Instruct
   ⚡ Enabling Flash Attention 2


Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████| 338/338 [00:00<00:00, 627.77it/s, Materializing param=model.norm.weight]


   ✓ Model loaded successfully!

🔧 Fine-tuning configuration:
   Output dir: models/huggingface/training
   Epochs: 5
   Batch size: 4
   Gradient accumulation: 2
   Effective batch size: 8
   Learning rate: 0.0002
   Max sequence length: 2048
📝 Formatting 2275 training examples...
📝 Formatting 597 eval examples...
🔤 Creating and tokenizing datasets...


Tokenizing eval data: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 597/597 [00:00<00:00, 9172.93 examples/s]
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


   ✓ Datasets prepared

🚀 Starting fine-tuning...



Step,Training Loss,Validation Loss
100,0.534028,0.527378
200,0.514666,0.492676
300,0.409029,0.436398
400,0.294081,0.335029
500,0.211501,0.221489
600,0.123368,0.180649
700,0.099911,0.148635
800,0.094958,0.133967
900,0.066483,0.127302
1000,0.066324,0.129360


Writing model shards: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:04<00:00,  4.59s/it]



✅ Fine-tuning complete! Model saved to models/huggingface/training/final


PosixPath('models/huggingface/training/final')

In [1]:
from transformers import AutoTokenizer

# Load the Qwen tokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

# Save it to your model directory
tokenizer.save_pretrained("models/training/final")

print("✓ Tokenizer saved!")

✓ Tokenizer saved!
